# Cross-Tract Validation: Random Forest vs XGBoost

Train on one tract, predict the other. Each tract's overlay and features are
computed in complete isolation, **no shared context between train and test**.

| Run | Model | Train on | Predict on |
|-----|-------|----------|------------|
| 1 | Random Forest | Tract 1808 | Tract 1219 |
| 2 | Random Forest | Tract 1219 | Tract 1808 |
| 3 | XGBoost | Tract 1808 | Tract 1219 |
| 4 | XGBoost | Tract 1219 | Tract 1808 |


In [1]:
import os, warnings, time
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

OBM_NJ            = "/scratch/gpfs/GVILLARI/hackathon_2026/FULL_NJ/obm_nj.gpkg"
LABELS_1808       = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/cleaned_lookuptable.gpkg"
LABELS_1219       = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/1219_CLEAN_PARCELS.gpkg"
UNCLEANED_PARCELS = "/scratch/gpfs/GVILLARI/hackathon_2026/TOY_PROBLEM/spatial_join_for_cleaning.gpkg"
OUT_BASE          = "/scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_cross_tract_validation"

run_ts  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_RUN = os.path.join(OUT_BASE, run_ts)
os.makedirs(OUT_RUN, exist_ok=True)
print(f"Output folder: {OUT_RUN}")

N_ESTIMATORS       = 200
RANDOM_STATE       = 42
STRONG_CAND_THRESH = 0.3
CONFIDENCE_THRESH  = 0.8

FEATURES = [
    "fp_area", "fp_perim", "fp_compact",
    "parcel_area", "parcel_perim",
    "centroid_dist", "area_ratio", "area_diff",
    "overlap_frac", "piece_area", "frac_of_best",
    "n_fps_for_parcel", "n_parcels_for_fp",
    "overlap_rank", "area_rank",
    "fp_split_entropy", "fp_max_overlap", "fp_is_best_parcel",
    "fp_gap_from_best", "fp_abs_area_elsewhere",
    "fp_covers_parcel_frac", "parcel_total_candidate_coverage",
    "n_strong_candidates",
]


Output folder: /scratch/gpfs/GVILLARI/mb6477/parcelAI/outputs_cross_tract_validation/2026-06-03_10-22-01


## Labels

In [2]:
def load_labels(path):
    df = gpd.read_file(path)[["PAMS_PIN", "footprint_ids", "method"]]
    return df.dropna(subset=["method"])

def build_positive_set(labels_df):
    labels_df = labels_df.copy()
    labels_df["fp_id_list"] = labels_df["footprint_ids"].apply(
        lambda x: x.split(",") if isinstance(x, str) else [])
    pairs = labels_df[["PAMS_PIN", "fp_id_list"]].explode("fp_id_list")
    pairs = pairs[pairs["fp_id_list"].notna() & (pairs["fp_id_list"] != "")]
    pairs.columns = ["PAMS_PIN", "fp_id"]
    pairs["fp_id"] = pairs["fp_id"].astype(str).str.strip()
    return set(zip(pairs["PAMS_PIN"], pairs["fp_id"]))

labels_1808  = load_labels(LABELS_1808)
labels_1219  = load_labels(LABELS_1219)
pins_1808    = set(labels_1808["PAMS_PIN"])
pins_1219    = set(labels_1219["PAMS_PIN"])
pos_set_1808 = build_positive_set(labels_1808)
pos_set_1219 = build_positive_set(labels_1219)

print(f"Tract 1808: {len(pins_1808):,} parcels  {len(pos_set_1808):,} positive pairs")
print(f"Tract 1219: {len(pins_1219):,} parcels  {len(pos_set_1219):,} positive pairs")
print(f"PIN overlap between tracts: {len(pins_1808 & pins_1219)}  (must be 0)")


Tract 1808: 748 parcels  763 positive pairs
Tract 1219: 600 parcels  520 positive pairs
PIN overlap between tracts: 0  (must be 0)


## Overlays

Each tract's overlay and features are computed independently.
Tract 1808 knows nothing about tract 1219's parcels and vice versa.


In [3]:
def build_tract_overlay(pins, positive_set, label):
    """Full pipeline for one tract: parcels → OBM → overlay → 23 features → labels."""
    print(f"\nBuilding overlay for tract {label}...")

    # parcel geometries
    parcels = gpd.read_file(
        UNCLEANED_PARCELS, columns=["PAMS_PIN", "geometry"]
    ).to_crs("EPSG:3424")
    parcels = parcels[parcels["PAMS_PIN"].isin(pins)].copy().reset_index(drop=True)
    parcels["parcel_area"]       = parcels.geometry.area
    parcels["parcel_perim"]      = parcels.geometry.length
    parcels["parcel_centroid_x"] = parcels.geometry.centroid.x
    parcels["parcel_centroid_y"] = parcels.geometry.centroid.y
    print(f"  {len(parcels):,} parcels")

    # OBM footprints clipped to this tract only
    obm = gpd.read_file(OBM_NJ, bbox=tuple(parcels.total_bounds)).to_crs("EPSG:3424")
    obm = obm.drop_duplicates(subset=["id"]).reset_index(drop=True)
    obm["fp_area"]       = obm.geometry.area
    obm["fp_perim"]      = obm.geometry.length
    obm["fp_compact"]    = (4 * np.pi * obm["fp_area"]) / (obm["fp_perim"] ** 2)
    obm["fp_centroid_x"] = obm.geometry.centroid.x
    obm["fp_centroid_y"] = obm.geometry.centroid.y
    obm["id"]            = obm["id"].astype(str).str.strip()
    print(f"  {len(obm):,} footprints")

    # overlay
    t0 = time.time()
    ov = gpd.overlay(
        obm[["id","geometry","fp_area","fp_perim","fp_compact",
             "fp_centroid_x","fp_centroid_y"]],
        parcels[["PAMS_PIN","geometry","parcel_area","parcel_perim",
                 "parcel_centroid_x","parcel_centroid_y"]],
        how="intersection")
    ov["id"] = ov["id"].astype(str).str.strip()
    print(f"  {len(ov):,} candidate pairs  ({time.time()-t0:.1f}s)")

    # ── features ─────────────────────────────────────────────────────────────
    ov["piece_area"]    = ov.geometry.area
    ov["overlap_frac"]  = ov["piece_area"] / ov["fp_area"]
    ov["area_ratio"]    = ov["fp_area"]  / ov["parcel_area"]
    ov["area_diff"]     = (ov["fp_area"] - ov["parcel_area"]).abs()
    ov["centroid_dist"] = np.sqrt(
        (ov["parcel_centroid_x"] - ov["fp_centroid_x"])**2 +
        (ov["parcel_centroid_y"] - ov["fp_centroid_y"])**2)
    ov["n_fps_for_parcel"] = ov.groupby("PAMS_PIN")["id"].transform("count")
    ov["n_parcels_for_fp"] = ov.groupby("id")["PAMS_PIN"].transform("count")
    ov["overlap_rank"]     = ov.groupby("PAMS_PIN")["overlap_frac"].rank(ascending=False)
    ov["area_rank"]        = ov.groupby("PAMS_PIN")["fp_area"].rank(ascending=False)
    ov["frac_of_best"]     = ov["overlap_frac"] / ov.groupby("PAMS_PIN")["overlap_frac"].transform("max")
    ov["_fr"]  = ov["overlap_frac"].clip(lower=1e-9)
    _p         = ov["_fr"] / ov.groupby("id")["_fr"].transform("sum")
    ov["_plogp"]           = -(_p * np.log(_p))
    ov["fp_split_entropy"] = ov.groupby("id")["_plogp"].transform("sum")
    ov.drop(columns=["_fr","_plogp"], inplace=True)
    ov["fp_max_overlap"]        = ov.groupby("id")["overlap_frac"].transform("max")
    ov["fp_is_best_parcel"]     = (ov["overlap_frac"] == ov["fp_max_overlap"]).astype(int)
    ov["fp_gap_from_best"]      = ov["fp_max_overlap"] - ov["overlap_frac"]
    ov["fp_abs_area_elsewhere"] = ov["fp_area"] * (1.0 - ov["overlap_frac"])
    ov["fp_covers_parcel_frac"] = ov["piece_area"] / ov["parcel_area"]
    ov["parcel_total_candidate_coverage"] = (
        ov.groupby("PAMS_PIN")["piece_area"].transform("sum") / ov["parcel_area"])
    ov["_above"]              = (ov["overlap_frac"] > STRONG_CAND_THRESH).astype("int8")
    ov["n_strong_candidates"] = ov.groupby("PAMS_PIN")["_above"].transform("sum")
    ov.drop(columns=["_above"], inplace=True)

    # ── labels ────────────────────────────────────────────────────────────────
    _keys = list(zip(ov["PAMS_PIN"].to_numpy(), ov["id"].to_numpy()))
    ov["correct"] = np.fromiter(
        (1 if k in positive_set else 0 for k in _keys),
        count=len(_keys), dtype="int8")

    print(f"  positives: {ov['correct'].sum():,}  negatives: {(ov['correct']==0).sum():,}")
    return ov, obm

ov_1808, obm_1808 = build_tract_overlay(pins_1808, pos_set_1808, "1808")
ov_1219, obm_1219 = build_tract_overlay(pins_1219, pos_set_1219, "1219")



Building overlay for tract 1808...
  748 parcels
  2,131 footprints
  818 candidate pairs  (0.0s)
  positives: 763  negatives: 55

Building overlay for tract 1219...
  600 parcels
  3,716 footprints
  850 candidate pairs  (0.0s)
  positives: 520  negatives: 330


## Models

In [4]:
def make_rf():
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

def make_xgb(y_train):
    n_pos = int(y_train.sum())
    n_neg = int((y_train == 0).sum())
    return XGBClassifier(
        n_estimators     = N_ESTIMATORS,
        learning_rate    = 0.1,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = n_neg / max(n_pos, 1),
        device           = "cuda",
        objective        = "binary:logistic",
        eval_metric      = "logloss",
        random_state     = RANDOM_STATE,
        verbosity        = 0,
    )

def evaluate(ov_test, label):
    y_test = ov_test["correct"]
    proba  = ov_test["confidence"]
    pred   = ov_test["predicted"]

    p, r, f1, _ = precision_recall_fscore_support(
        y_test, pred, labels=[1], zero_division=0)

    # parcel-level: did the model find the true match for each parcel?
    parcel_stats = ov_test.groupby("PAMS_PIN").apply(lambda g: pd.Series({
        "has_true_match":      int((g["correct"] == 1).any()),
        "true_match_found":    int((g.loc[g["correct"]==1, "predicted"]==1).any())
                               if (g["correct"]==1).any() else 0,
        "top_pred_is_correct": int(g.loc[g["confidence"].idxmax(), "correct"]==1),
    }), include_groups=False)

    with_match    = parcel_stats[parcel_stats["has_true_match"] == 1]
    parcel_recall = with_match["true_match_found"].mean()
    top1_acc      = parcel_stats["top_pred_is_correct"].mean()

    print(f"\n{'='*52}")
    print(f"  {label}")
    print(f"{'='*52}")
    print(f"  Pairs  — Precision: {float(p[0]):.3f}  Recall: {float(r[0]):.3f}  F1: {float(f1[0]):.3f}")
    print(f"  Parcel recall:   {parcel_recall:.3f}  ({int(with_match['true_match_found'].sum())}/{len(with_match)} parcels)")
    print(f"  Top-1 accuracy:  {top1_acc:.3f}")

    return {
        "label":          label,
        "pair_precision": float(p[0]),
        "pair_recall":    float(r[0]),
        "pair_f1":        float(f1[0]),
        "parcel_recall":  float(parcel_recall),
        "top1_accuracy":  float(top1_acc),
        "n_test_pairs":   len(y_test),
        "n_test_parcels": len(parcel_stats),
    }


## Run all 4 folds

In [5]:
results = []

tract_data = {
    "1808": {"ov": ov_1808, "obm": obm_1808},
    "1219": {"ov": ov_1219, "obm": obm_1219},
}

folds = [
    ("RF",      "1808", "1219"),
    ("RF",      "1219", "1808"),
    ("XGBoost", "1808", "1219"),
    ("XGBoost", "1219", "1808"),
]

for model_name, train_tract, test_tract in folds:
    label     = f"{model_name}  train={train_tract}  test={test_tract}"
    fold_slug = f"{model_name.lower()}_train{train_tract}_test{test_tract}"
    print(f"\nRunning: {label}")

    ov_train = tract_data[train_tract]["ov"]
    ov_test  = tract_data[test_tract]["ov"].copy()
    obm_test = tract_data[test_tract]["obm"]

    X_train = ov_train[FEATURES].fillna(0)
    y_train = ov_train["correct"]
    X_test  = ov_test[FEATURES].fillna(0)

    # train
    t0 = time.time()
    if model_name == "RF":
        m = make_rf()
        m.fit(X_train, y_train)
    else:
        m = make_xgb(y_train)
        m.fit(X_train.astype("float32"), y_train.astype("int32"))
    print(f"  trained in {time.time()-t0:.1f}s")

    # predict
    ov_test["confidence"]   = m.predict_proba(X_test)[:, 1]
    ov_test["predicted"]    = (ov_test["confidence"] >= 0.5).astype(int)
    ov_test["needs_review"] = ov_test["confidence"] < CONFIDENCE_THRESH

    # evaluate
    res = evaluate(ov_test, label)
    results.append(res)

    # ── save outputs ──────────────────────────────────────────────────────────
    t1 = time.time()

    # predicted_assignments.gpkg
    assigned = (
        ov_test[ov_test["predicted"] == 1]
        .sort_values("confidence", ascending=False)
        .groupby("PAMS_PIN")
        .agg(predicted_fp_ids=("id",         lambda x: ",".join(x)),
             n_predicted      =("id",         "count"),
             avg_confidence   =("confidence", "mean"))
        .reset_index()
    )
    assigned["needs_review"] = assigned["avg_confidence"] < CONFIDENCE_THRESH

    test_parcels = gpd.read_file(
        UNCLEANED_PARCELS, columns=["PAMS_PIN","geometry"]
    ).to_crs("EPSG:3424")
    test_parcels = test_parcels[
        test_parcels["PAMS_PIN"].isin(ov_test["PAMS_PIN"].unique())
    ]
    parcel_out = test_parcels.merge(assigned, on="PAMS_PIN", how="left")
    parcel_out["n_predicted"]  = parcel_out["n_predicted"].fillna(0).astype(int)
    parcel_out["needs_review"] = parcel_out["needs_review"].fillna(False)

    gpd.GeoDataFrame(parcel_out, geometry="geometry", crs="EPSG:3424").to_file(
        os.path.join(OUT_RUN, f"{fold_slug}_predicted_assignments.gpkg"), driver="GPKG")

    # predicted_footprints.gpkg
    fp_out = (
        ov_test[ov_test["predicted"] == 1][[
            "PAMS_PIN","id","confidence","needs_review","correct",
            "overlap_frac","piece_area","centroid_dist",
            "fp_split_entropy","frac_of_best"
        ]].merge(obm_test[["id","geometry"]], on="id", how="left")
    )
    gpd.GeoDataFrame(fp_out, geometry="geometry", crs="EPSG:3424").to_file(
        os.path.join(OUT_RUN, f"{fold_slug}_predicted_footprints.gpkg"), driver="GPKG")

    print(f"  saved in {time.time()-t1:.1f}s")
    print(f"    {fold_slug}_predicted_assignments.gpkg  ({len(parcel_out):,} parcels)")
    print(f"    {fold_slug}_predicted_footprints.gpkg   ({len(fp_out):,} pairs)")



Running: RF  train=1808  test=1219
  trained in 0.3s

  RF  train=1808  test=1219
  Pairs  — Precision: 0.903  Recall: 0.983  F1: 0.941
  Parcel recall:   0.988  (510/516 parcels)
  Top-1 accuracy:  0.916
  saved in 0.2s
    rf_train1808_test1219_predicted_assignments.gpkg  (559 parcels)
    rf_train1808_test1219_predicted_footprints.gpkg   (566 pairs)

Running: RF  train=1219  test=1808
  trained in 0.2s

  RF  train=1219  test=1808
  Pairs  — Precision: 0.997  Recall: 0.896  F1: 0.944
  Parcel recall:   0.991  (680/686 parcels)
  Top-1 accuracy:  0.981
  saved in 0.2s
    rf_train1219_test1808_predicted_assignments.gpkg  (699 parcels)
    rf_train1219_test1808_predicted_footprints.gpkg   (686 pairs)

Running: XGBoost  train=1808  test=1219
  trained in 0.9s

  XGBoost  train=1808  test=1219
  Pairs  — Precision: 0.951  Recall: 0.963  F1: 0.957
  Parcel recall:   0.969  (500/516 parcels)
  Top-1 accuracy:  0.912
  saved in 0.2s
    xgboost_train1808_test1219_predicted_assignments.gpk

## Results

In [ ]:
summary = pd.DataFrame(results).set_index("label").rename(columns={
    "pair_precision": "Precision",
    "pair_recall":    "Recall",
    "pair_f1":        "F1",
    "parcel_recall":  "Parcel recall",
    "top1_accuracy":  "Top-1 accuracy",
    "n_test_pairs":   "Test pairs",
    "n_test_parcels": "Test parcels",
})

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(summary.to_string(float_format="{:.3f}".format))

print("\nBest per metric:")
for col in ["Precision","Recall","F1","Parcel recall","Top-1 accuracy"]:
    best = summary[col].idxmax()
    print(f"  {col:<16} {best}  ({summary.loc[best, col]:.3f})")

summary.to_csv(os.path.join(OUT_RUN, "summary.csv"))
print(f"\nAll outputs in: {OUT_RUN}")


## Summary

Training on **messy (1219)** data produces a conservative model - near-perfect precision
(0.997–0.999) but misses ~10% of real matches. Training on **clean (1808)** data produces
a recall-biased model - catches more matches but over-predicts on harder cases (precision
0.903–0.951). Both models generalize well overall (F1 > 0.94, parcel recall > 0.97).

> Harder training data → more robust, precise predictions at deployment